# 테스트 셋 평가 전용 노트북

**Colored MNIST - 학습된 모델 평가**

이 노트북은 이미 학습된 모델을 로드하여 교수님의 테스트 셋만으로 평가합니다.

## 사용 방법

1. **테스트 셋 파일 준비**
   - 교수님이 제공한 테스트 셋 파일을 `data/test/professor_test_set.npz`에 배치
   - 또는 `configs/paths.yaml`의 `test` 경로를 수정

2. **필요한 파일 확인**
   - `results/models/` 디렉토리에 학습된 모델 파일(.joblib)이 있어야 함
   - `configs/models.yaml` 파일 필요

3. **실행**
   - 셀을 순서대로 실행
   - 결과는 `results/metrics/test_*.csv`와 `results/figures/cm_*_test_*.png`에 저장됨

In [ ]:
# ==========================================
# Cell 1. 환경 설정 및 라이브러리 임포트
# ==========================================
import os
import numpy as np
import pandas as pd
from pathlib import Path
import yaml
import joblib
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import (
    accuracy_score,
    precision_recall_fscore_support,
    classification_report,
    confusion_matrix
)

# 재현성
RANDOM_SEED = 0
np.random.seed(RANDOM_SEED)

# Matplotlib 설정
plt.rcParams["font.family"] = "DejaVu Sans"
plt.rcParams["axes.unicode_minus"] = False

print("[INFO] 라이브러리 임포트 완료")

In [ ]:
# ==========================================
# Cell 2. Config 파일 로드
# ==========================================
def load_config(config_path):
    """YAML config 파일 로드"""
    notebook_dir = Path.cwd()
    if 'notebooks' in str(notebook_dir):
        project_root = notebook_dir.parent
    else:
        project_root = notebook_dir
    
    config_file = Path(config_path)
    if not config_file.is_absolute():
        config_file = project_root / config_path
    
    with open(config_file, 'r', encoding='utf-8') as f:
        return yaml.safe_load(f)

def get_paths(config_path="configs/paths.yaml"):
    """경로 config 로드 및 절대 경로 변환"""
    config = load_config(config_path)
    paths = config["paths"]
    
    notebook_dir = Path.cwd()
    if 'notebooks' in str(notebook_dir):
        project_root = notebook_dir.parent
    else:
        project_root = notebook_dir
    
    resolved = {}
    for key, value in paths.items():
        if isinstance(value, dict):
            resolved[key] = {
                k: project_root / v if v is not None else None
                for k, v in value.items()
            }
        else:
            resolved[key] = project_root / value if value is not None else None
    return resolved

# 경로 로드
paths = get_paths("configs/paths.yaml")
print(f"[INFO] Config 로드 완료")
print(f"[INFO] Test 셋 경로: {paths['data']['test']}")

In [ ]:
# ==========================================
# Cell 3. 테스트 셋 로드
# ==========================================
test_path = paths['data']['test']

if test_path is None or not test_path.exists():
    print(f"[ERROR] 테스트 셋 파일을 찾을 수 없습니다: {test_path}")
    print(f"[INFO] configs/paths.yaml의 test 경로를 확인하거나, 테스트 셋 파일을 준비해주세요.")
    raise FileNotFoundError(f"Test set not found: {test_path}")

# 테스트 셋 로드
test_data = np.load(test_path, allow_pickle=True)

# 테스트 셋 구조 확인
print("[INFO] 테스트 셋 키:", list(test_data.keys()))

# 테스트 데이터 추출
# 예상 구조: X_test, y_digit_test, y_fg_test, y_bg_test
X_test = test_data["X_test"]
y_digit_test = test_data.get("y_digit_test", None)
y_fg_test = test_data.get("y_fg_test", None)
y_bg_test = test_data.get("y_bg_test", None)

print(f"[INFO] X_test shape: {X_test.shape}")
if y_digit_test is not None:
    print(f"[INFO] y_digit_test shape: {y_digit_test.shape}, classes: {np.unique(y_digit_test)}")
if y_fg_test is not None:
    print(f"[INFO] y_fg_test shape: {y_fg_test.shape}, classes: {np.unique(y_fg_test)}")
if y_bg_test is not None:
    print(f"[INFO] y_bg_test shape: {y_bg_test.shape}, classes: {np.unique(y_bg_test)}")

In [ ]:
# ==========================================
# Cell 4. 평가 함수 정의
# ==========================================
def compute_metrics(y_true, y_pred, average="macro"):
    """성능 지표 계산"""
    acc = accuracy_score(y_true, y_pred)
    prec, rec, f1, _ = precision_recall_fscore_support(
        y_true, y_pred, average=average, zero_division=0
    )
    return acc, prec, rec, f1

def plot_confusion_matrix(y_true, y_pred, classes, title, save_path, normalize=False):
    """Confusion Matrix 시각화"""
    cm = confusion_matrix(y_true, y_pred, labels=classes)
    
    if normalize:
        cm = cm.astype('float') / cm.sum(axis=1)[:, np.newaxis]
        fmt = '.2f'
    else:
        fmt = 'd'
    
    plt.figure(figsize=(10, 8))
    sns.heatmap(cm, annot=True, fmt=fmt, cmap='Blues', 
                xticklabels=classes, yticklabels=classes)
    plt.title(title)
    plt.ylabel('True Label')
    plt.xlabel('Predicted Label')
    plt.tight_layout()
    plt.savefig(save_path, dpi=150, bbox_inches='tight')
    plt.close()
    print(f"[INFO] Confusion Matrix 저장: {save_path}")

print("[INFO] 평가 함수 정의 완료")

In [ ]:
# ==========================================
# Cell 5. Task 및 모델 설정
# ==========================================
# 평가할 Task
ACTIVE_TASKS = ["digit", "fg", "bg"]

# 평가할 모델
ACTIVE_MODELS = ["knn", "svm", "tree", "rf", "xgb"]

# Task별 라벨 매핑
TASK_LABELS = {
    "digit": y_digit_test,
    "fg": y_fg_test,
    "bg": y_bg_test
}

print(f"[INFO] 평가할 Task: {ACTIVE_TASKS}")
print(f"[INFO] 평가할 모델: {ACTIVE_MODELS}")

In [ ]:
# ==========================================
# Cell 6. 모델 로드 및 테스트 셋 평가
# ==========================================
MODELS_DIR = paths["results"]["models"]
FIGURES_DIR = paths["results"]["figures"]
METRICS_DIR = paths["results"]["metrics"]

# 결과 저장용
test_results = []

# 각 Task별로 평가
for task_name in ACTIVE_TASKS:
    y_test = TASK_LABELS[task_name]
    
    if y_test is None:
        print(f"[WARN] {task_name} Task의 라벨이 없습니다. 건너뜁니다.")
        continue
    
    print(f"\n{'='*60}")
    print(f"### Test Set Evaluation: {task_name}")
    print(f"{'='*60}")
    
    # 각 모델별 평가
    for model_name in ACTIVE_MODELS:
        # 모델 파일 경로
        model_path = MODELS_DIR / f"{task_name}_{model_name}.joblib"
        
        if not model_path.exists():
            print(f"[WARN] 모델 파일이 없습니다: {model_path}")
            continue
        
        print(f"\n[INFO] 모델 로드: {model_path}")
        
        # 모델 로드 (Pipeline 포함)
        pipe = joblib.load(model_path)
        
        # 테스트 셋 예측
        y_test_pred = pipe.predict(X_test)
        
        # 성능 평가
        test_acc, test_prec, test_rec, test_f1 = compute_metrics(y_test, y_test_pred, average="macro")
        
        print(f"\n[TEST] Macro metrics ({task_name}, {model_name})")
        print(f"  accuracy : {test_acc:.4f} ({test_acc*100:.2f}%)")
        print(f"  precision: {test_prec:.4f}")
        print(f"  recall   : {test_rec:.4f}")
        print(f"  f1-score : {test_f1:.4f}")
        
        # Classification Report
        print(f"\n[TEST] Classification Report:")
        print(classification_report(y_test, y_test_pred, digits=4))
        
        # Confusion Matrix 저장
        classes = np.sort(np.unique(y_test))
        
        # Count 버전
        cm_path_count = FIGURES_DIR / f"cm_{task_name}_{model_name}_test_counts.png"
        plot_confusion_matrix(
            y_true=y_test,
            y_pred=y_test_pred,
            classes=classes,
            title=f"{task_name} - {model_name} (Test Set, Counts)",
            save_path=str(cm_path_count),
            normalize=False
        )
        
        # Normalized 버전
        cm_path_norm = FIGURES_DIR / f"cm_{task_name}_{model_name}_test_normalized.png"
        plot_confusion_matrix(
            y_true=y_test,
            y_pred=y_test_pred,
            classes=classes,
            title=f"{task_name} - {model_name} (Test Set, Normalized)",
            save_path=str(cm_path_norm),
            normalize=True
        )
        
        # 결과 저장
        test_results.append({
            "task": task_name,
            "model": model_name,
            "test_accuracy": test_acc,
            "test_precision": test_prec,
            "test_recall": test_rec,
            "test_f1": test_f1
        })

# 결과 요약
if test_results:
    test_results_df = pd.DataFrame(test_results)
    test_results_df = test_results_df.sort_values(
        by=["task", "test_accuracy"],
        ascending=[True, False]
    ).reset_index(drop=True)
    
    # CSV 저장
    test_summary_path = METRICS_DIR / "test_metrics_all_models.csv"
    test_results_df.to_csv(test_summary_path, index=False)
    
    print(f"\n{'='*60}")
    print("### Test Set Results Summary")
    print(f"{'='*60}")
    print(test_results_df.to_string())
    print(f"\n[INFO] 결과 저장: {test_summary_path}")
else:
    print("[WARN] 평가된 결과가 없습니다.")